# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout (`holdout.csv`), stratified 5-fold CV on training rows only, model chosen by mean CV roc_auc, holdout scored once.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()   # 0-based positions in data.csv
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'
assert len(df)==7000 and len(set(hold_idx))==1400
is_hold = np.zeros(len(df), bool); is_hold[hold_idx] = True
train_idx = np.where(~is_hold)[0]
print(len(train_idx), len(hold_idx))

## Features
`device_serial` is excluded (identifier, per protocol). The rest — channel, device age at sale, price, daily hours reported at registration — are all known when the warranty is sold, so none leak the outcome.

In [ ]:
target='claim_within_year'
cat=['channel']; num=['device_age_years','price_eur','daily_hours']
features=cat+num
X=df[features]; y=df[target]
# training-only exploration
print(y.iloc[train_idx].mean())
print(df.iloc[train_idx].groupby('channel')[target].agg(['mean','size']))
print(df.iloc[train_idx][num+[target]].corr()[target])

## Candidates (named before any scoring)
1. `baseline_prior` – predicts class prior
2. `logreg` – scaled numerics + one-hot channel
3. `random_forest`
4. `hist_gb` – gradient boosting

Preprocessing lives inside each pipeline, so it is refit in every CV fold.

In [ ]:
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),
                              ('n',StandardScaler() if scale else 'passthrough',num)])
candidates = {
 'baseline_prior': lambda: Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg': lambda: Pipeline([('p',prep(True)),('m',LogisticRegression(max_iter=1000))]),
 'random_forest': lambda: Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=1))]),
 'hist_gb': lambda: Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

## Cross-validation on training rows

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=0)
Xtr, ytr = X.iloc[train_idx], y.iloc[train_idx]
splits = list(skf.split(Xtr, ytr))
cv_folds = [[[int(i) for i in train_idx[a]], [int(i) for i in train_idx[b]]] for a,b in splits]  # [train, test] 0-based rows of data.csv
cv_scores = {}
for name,mk in candidates.items():
    s=[]
    for a,b in splits:
        m=mk().fit(Xtr.iloc[a],ytr.iloc[a])
        s.append(float(roc_auc_score(ytr.iloc[b],m.predict_proba(Xtr.iloc[b])[:,1])))
    cv_scores[name]=s
print(pd.DataFrame({n:{'mean':np.mean(s),'std':np.std(s)} for n,s in cv_scores.items()}).T)
chosen = max(cv_scores, key=lambda n: np.mean(cv_scores[n]))
print('chosen:', chosen)

## Holdout (scored once)

In [ ]:
final=candidates[chosen]().fit(Xtr,ytr)
holdout_auc=float(roc_auc_score(y.iloc[hold_idx],final.predict_proba(X.iloc[hold_idx])[:,1]))
print(f'Holdout roc_auc ({chosen}): {holdout_auc:.4f}')

In [ ]:
evidence={'features':features,'candidates':list(candidates),'cv_folds':cv_folds,'cv_scores':cv_scores,
 'chosen':chosen,'holdout_rows':[int(i) for i in hold_idx],'holdout_scores':{chosen:holdout_auc}}
json.dump(evidence,open('evidence.json','w'))